In [2]:
import ollama
import pymupdf  # pip install pymupdf

def summarize_pdf(pdf_path: str, model: str = "llama3.2") -> str:
    doc = pymupdf.open(pdf_path)
    text = "\n".join(page.get_text() for page in doc)
    
    # Chunk if too long (models have context limits)
    max_chars = 12000
    text = text[:max_chars]

    response = ollama.chat(
        model=model,
        messages=[{
            "role": "user",
            "content": f"Summarize this research paper concisely:\n\n{text}"
        }]
    )
    return response["message"]["content"]

In [3]:
summarize_pdf('2606.pdf')

'Here is a concise summary of the research paper:\n\nThe study examines whether prediction markets match option prices, using Bitcoin as an example. The authors compare the prices of Polymarket Yes contracts with those of Binance call options for identical underlying assets (Bitcoin) and maturities (September 2023). They find a statistically significant positive wedge between the two prices, indicating that Polymarket Yes prices are consistently higher than Binance-implied risk-neutral values.\n\nThe study also finds that the gap between the two prices is persistent over intraday horizons with an autoregressive half-life of approximately four hours. This suggests that information and capital move across the two ecosystems but not instantaneously and not without cost.\n\nCross-sectional regressions reveal that the wedge is largest at low option-implied probabilities and long maturities, consistent with speculative demand for out-of-the-money prediction-market contracts.\n\nThe study con

# Scraping publication from the Arvix Website

Scrape Arvix and score author to determine crediblility

In [1]:
from scrape_credibility import get_score

In [3]:
get_score('Revisiting Trade-sign Long-memory and Square-root Law price impact')

35

In [5]:
import arxiv

def get_latest_microstructure_papers(n: int = 100) -> list[dict]:
    """
    Fetch the latest `n` papers from arXiv q-fin.TR
    (Trading and Market Microstructure), returning title + authors.
    """
    client = arxiv.Client(page_size=n, delay_seconds=1)
    search = arxiv.Search(
        query="cat:q-fin.TR",
        max_results=n,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )

    results = []
    for paper in client.results(search):
        results.append({
            "title": paper.title,
            "published": paper.published.date().isoformat(),
            "authors": [str(a) for a in paper.authors],
            "arxiv_id": paper.get_short_id(),
            "url": paper.entry_id,
        })

    return results


papers = get_latest_microstructure_papers(n=100)
print(f"Fetched {len(papers)} papers\n")
# for p in papers:
#     print(f"[{p['published']}] {p['title']}")
#     print(f"  Authors: {', '.join(p['authors'])}")
#     print(f"  {p['url']}\n")

Fetched 100 papers



In [9]:
succes = 0
total = 0

for paper in papers:
    if isinstance(get_score(paper['title']), int):
        succes += 1
    total+=1

[ERROR] OpenAlex works API returned 503
[ERROR] OpenAlex works API returned 503
[ERROR] OpenAlex works API returned 503
[ERROR] OpenAlex works API returned 503
[ERROR] OpenAlex works API returned 400
[ERROR] OpenAlex works API returned 503
[ERROR] OpenAlex works API returned 503
[ERROR] OpenAlex works API returned 400
[ERROR] OpenAlex works API returned 503
[ERROR] OpenAlex works API returned 400
[ERROR] OpenAlex works API returned 400
[ERROR] OpenAlex works API returned 503
[ERROR] OpenAlex works API returned 400
[ERROR] OpenAlex works API returned 400
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429
[ERROR] OpenAlex works API returned 429


In [10]:
succes

67

In [24]:
import os
from dotenv import dotenv_values

# Bypass os.environ entirely — reads .env directly
config = dotenv_values()
# print(config)  # should show {'GOOGLE_API_KEY': '...'}

# Then set it manually
os.environ["GOOGLE_API_KEY"] = config["GOOGLE_API_KEY"]

In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.7
)

# Ask a question
response = llm.invoke("Explain recursion in simple terms")
response

AIMessage(content='Imagine you have a set of Russian nesting dolls. You open the largest doll, and inside you find a slightly smaller doll. You open that one, and find an even smaller doll, and so on. You keep opening dolls until you find the tiniest, solid doll that can\'t be opened anymore.\n\n**Recursion is like that with tasks or problems.**\n\nHere\'s the simple breakdown:\n\n1.  **A Self-Referential Task:** Recursion is when a task or a function (a set of instructions) solves a problem by calling **itself** to solve a smaller, identical version of the same problem.\n\n2.  **The "Smaller Version":** Just like the nesting dolls get smaller, each recursive call works on a simpler or reduced version of the original problem.\n\n3.  **The "Base Case" (The Tiny Doll):** Crucially, there has to be a point where the problem is so simple that it can be solved directly, without needing to call itself again. This is the **base case**. It\'s the stopping condition. Without a base case, the re